# 04 — Bivariate Analysis
## Guided Jupyter Notebook

Bivariate analysis explores how two variables vary together. This notebook covers numerical–numerical, numerical–categorical, and categorical–categorical pairs.

**Dataset:** synthetic student-performance records generated below. No external download is needed.


## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()
pd.set_option("display.max_columns", None)


## 2. Create a reproducible dataset
The synthetic data includes several numerical and categorical variables. The results are for learning analysis methods, not real-world conclusions.

In [ ]:
rng = np.random.default_rng(24)
n = 180

department = rng.choice(
    ["Computer Science", "Business", "Engineering", "Arts"],
    size=n, p=[0.36, 0.22, 0.29, 0.13]
)
study_hours = np.clip(rng.gamma(shape=3.0, scale=2.8, size=n), 0.5, 25)
attendance = np.clip(rng.normal(82, 10, size=n), 40, 100)
department_effect = pd.Series(department).map({
    "Computer Science": 3, "Business": 0, "Engineering": 2, "Arts": -2
}).to_numpy()
exam_score = np.clip(
    43 + 1.15 * study_hours + 0.22 * (attendance - 70)
    + department_effect + rng.normal(0, 10, size=n),
    0, 100
)

df = pd.DataFrame({
    "student_id": [f"STU-{i:03d}" for i in range(1, n + 1)],
    "study_hours": study_hours.round(1),
    "attendance_pct": attendance.round(1),
    "exam_score": exam_score.round(1),
    "department": department,
    "study_mode": rng.choice(["Self-study", "Online course", "Classroom", "Hybrid"], size=n),
    "scholarship": rng.choice(["Yes", "No"], size=n, p=[0.28, 0.72]),
    "satisfaction": rng.choice(
        ["Very dissatisfied", "Dissatisfied", "Neutral", "Satisfied", "Very satisfied"],
        size=n, p=[0.05, 0.10, 0.20, 0.40, 0.25]
    )
})

# Add missing values to demonstrate pairwise handling.
df.loc[rng.choice(df.index, 8, replace=False), "exam_score"] = np.nan
df.loc[rng.choice(df.index, 6, replace=False), "study_hours"] = np.nan
df.loc[rng.choice(df.index, 5, replace=False), "department"] = np.nan

df.head()


## 3. Inspect the data and missingness

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.dtypes.rename("dtype").to_frame())

missing = pd.DataFrame({
    "count": df.isna().sum(),
    "percent": df.isna().mean().mul(100).round(2)
})
display(missing)


## 4. Numerical vs numerical: scatter plots
Look for direction, form, clusters, and unusual points before interpreting a coefficient.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="study_hours", y="exam_score", ax=axes[0])
axes[0].set_title("Study Hours vs Exam Score")
axes[0].set_xlabel("Study hours per week")
axes[0].set_ylabel("Exam score")

sns.regplot(data=df, x="attendance_pct", y="exam_score", ax=axes[1],
            scatter_kws={"alpha": 0.6})
axes[1].set_title("Attendance vs Exam Score")
axes[1].set_xlabel("Attendance (%)")
axes[1].set_ylabel("Exam score")
plt.tight_layout()
plt.show()


## 5. Pearson, Spearman, and covariance
Use the same pairwise complete rows for both coefficients. Pearson describes linear association; Spearman describes monotonic rank association.

In [ ]:
pair = df[["study_hours", "exam_score"]].dropna()
print("Pairwise sample size:", len(pair))
print("Pearson correlation:", pair["study_hours"].corr(pair["exam_score"], method="pearson"))
print("Spearman correlation:", pair["study_hours"].corr(pair["exam_score"], method="spearman"))

attendance_pair = df[["attendance_pct", "exam_score"]].dropna()
display(attendance_pair.cov())


## 6. Numerical vs categorical: group summaries and box plots

In [ ]:
department_summary = (
    df.groupby("department", dropna=False)["exam_score"]
      .agg(count="count", mean="mean", median="median", std="std")
      .sort_values("median", ascending=False)
)
display(department_summary.round(2))

sns.boxplot(data=df, x="department", y="exam_score")
plt.title("Exam Score Distribution by Department")
plt.xlabel("Department")
plt.ylabel("Exam score")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 7. Compare study modes

In [ ]:
mode_summary = (
    df.groupby("study_mode")["exam_score"]
      .agg(count="count", mean="mean", median="median", std="std")
      .sort_values("median", ascending=False)
)
display(mode_summary.round(2))

sns.boxplot(data=df, x="study_mode", y="exam_score")
plt.title("Exam Scores by Study Mode")
plt.xlabel("Study mode")
plt.ylabel("Exam score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## 8. Categorical vs categorical: cross-tabulation
Counts and percentages answer different questions. Row percentages compare the distribution of scholarship status within each department.

In [ ]:
cross_counts = pd.crosstab(df["department"], df["scholarship"], dropna=False)
cross_row_percent = pd.crosstab(
    df["department"], df["scholarship"], normalize="index"
).mul(100).round(2)

display(cross_counts)
display(cross_row_percent)

cross_counts.plot(kind="bar", stacked=True)
plt.title("Scholarship Status by Department")
plt.xlabel("Department")
plt.ylabel("Number of students")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


## 9. Correlation matrix

In [ ]:
numeric_cols = ["study_hours", "attendance_pct", "exam_score"]
corr = df[numeric_cols].corr(method="pearson")
display(corr.round(2))

sns.heatmap(corr, annot=True, fmt=".2f", center=0, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 10. Explore a relationship by subgroup
This is an exploratory view. Different subgroup trends do not establish causal effects.

In [ ]:
g = sns.lmplot(
    data=df.dropna(subset=["study_hours", "exam_score", "department"]),
    x="study_hours", y="exam_score", col="department", col_wrap=2,
    height=3.5, ci=None, scatter_kws={"alpha": 0.65}
)
g.set_axis_labels("Study hours per week", "Exam score")
g.fig.suptitle("Study Hours vs Exam Score by Department", y=1.02)
plt.show()


## 11. Reflection

1. What direction and form do the numerical relationships appear to have?
2. Do Pearson and Spearman correlations differ?
3. Which department has the highest median score in this sample?
4. How do scholarship percentages vary by department?
5. What confounders or selection effects might influence the observed patterns?
6. Why do these results not prove causation?

Continue with `practice.md`, then check your approach using `solution.md`.
